In [0]:
# ========================================
# Silver Layer: Regions with Data Quality
# ========================================

from pyspark.sql.functions import col, trim, upper, current_timestamp, when, length

# Read from Bronze
df_bronze = spark.read.table("ecommerce.e_comm_bronze.tblcustomers")

print(f"Bronze record count: {df_bronze.count()}")

# ── Data Quality Step 1: Standardize and clean text ──
df_cleaned = df_bronze \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("first_name", trim(upper(col("first_name")))) \
    .withColumn("last_name", trim(upper(col("last_name")))) \
    .withColumn("email", trim(upper(col("email")))) \
    .withColumn("city", trim(upper(col("city")))) \
    .withColumn("state", trim(upper(col("state"))))

# ── Data Quality Step 2: Remove duplicates ──
df_deduped = df_cleaned.dropDuplicates(["customer_id"])
df_deduped.createOrReplaceTempView("vw_df_deduped")
print(f"After deduplication: {df_deduped.count()}")

# ── Data Quality Step 3: Add surrogate_key and data quality flag ──
df_with_dq = spark.sql("""
        select 
        row_number() over (order by customer_id) as customer_key,
        customer_id,
        first_name,
        last_name, 
        email,
        city,
        state,
        case 
            when first_name is null or first_name == "" then "first_name_is_null"        
            when last_name is null or last_name == "" then "last_name_is_null"        
            when email is null or email == "" then "email_is_null"        
            when city is null or city == "" then "city_is_null"        
            when state is null or state == "" then "state_is_null"        
            else "is_valid" end as dq_note ,
        current_timestamp() as load_ts
        from vw_df_deduped 
                       """)

# Show data quality summary
print("\n=== Data Quality Summary ===")
print(f"Total records: {df_with_dq.count()}")
print(f"Valid records: {df_with_dq.filter(col('dq_note') == "is_valid").count()}")
print(f"Invalid records: {df_with_dq.filter(col('dq_note') != "is_valid").count()}")

# Show sample of invalid records if any exist
invalid_records = df_with_dq.filter(col('dq_note') != "is_valid")
if invalid_records.count() > 0:
    print("\nSample invalid records:")
    invalid_records.show(25, truncate=False)

# Create temp view for querying
df_with_dq.createOrReplaceTempView("vw_products")

print("\n✅ Silver transformation complete with data quality checks")

In [0]:
#write data to a delta table
df_with_dq \
    .write \
        .format("delta") \
            .option("overwriteSchema", "true") \
                .mode("overwrite").saveAsTable("ecommerce.e_comm_silver.customers")